# Video 9: CrewAI Flows and Advanced Workflows

## What We Will Build
In this notebook, we will build an **Advanced CrewAI Flow** that demonstrates:
- CrewAI Flows (event-driven workflows)
- State management across flow steps
- Conditional execution and routing
- Combining crews and flows
- Error handling in flows
- Passing state between steps

By the end, you will have a dynamic workflow that adapts based on intermediate results.

## 1. Install & Import

In [ ]:
!pip install -q crewai crewai-tools langchain-openai python-dotenv

## 2. Imports & Setup

In [ ]:
from crewai import Agent, Task, Crew, Processfrom crewai.flow.flow import Flow, listen, startfrom crewai.flow.state import Statefrom langchain_openai import ChatOpenAIimport osfrom dotenv import load_dotenvload_dotenv()llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.7, api_key=os.getenv('OPENAI_API_KEY'))print('Setup complete')

## 3. Theory: CrewAI Flows

### What are Flows?
**Flows** are event-driven workflows that go beyond the standard Crew execution model. They allow:
- **Custom state management**: Define your own state schema
- **Conditional routing**: Make decisions based on intermediate results
- **Event listeners**: Trigger actions when events occur
- **Combining crews**: Use crews as steps within a larger flow

### Flow vs Crew
| Feature | Crew | Flow |
|---------|------|------|
| Structure | Agents + Tasks | Steps with state |
| Execution | Sequential/Hierarchical | Event-driven |
| State | Implicit | Explicit custom state |
| Routing | Limited | Full conditional control |
| Best for | Standard pipelines | Complex adaptive workflows |

### Key Decorators
- `@start()`: Marks the entry point of a flow
- `@listen(event)`: Triggers when a specific event/step completes

## 4. Define the Flow State

In [ ]:
from typing import Dict, List, Optionalfrom pydantic import BaseModelclass ResearchFlowState(BaseModel):    """Custom state for the research flow."""    topic: str = ""    research_quality: str = ""  # 'high', 'medium', 'low'    research_output: str = ""    analysis_output: str = ""    final_report: str = ""    errors: List[str] = []    retry_count: int = 0print('Flow state defined with Pydantic BaseModel')

## 5. Create the Flow Class

A Flow class defines the workflow steps using decorators.

In [ ]:
class ResearchFlow(Flow[ResearchFlowState]):    """An advanced research flow with conditional routing."""        @start()    def initialize_topic(self):        """Step 1: Initialize the research topic."""        print(f'\n[Flow] Step 1: Initializing topic...')        self.state.topic = 'The Future of Quantum Computing in 2025'        print(f'  Topic set: {self.state.topic}')        return self.state.topic        @listen(initialize_topic)    def conduct_research(self, topic):        """Step 2: Conduct research using a crew."""        print(f'\n[Flow] Step 2: Conducting research on: {topic}')                # Create a mini-crew for research        researcher = Agent(            role='Research Specialist',            goal=f'Research: {topic}',            backstory='An expert researcher.',            llm=llm,            verbose=False        )                task = Task(            description=f'Research {topic} and provide key findings.',            expected_output='Research findings with key points.',            agent=researcher        )                crew = Crew(agents=[researcher], tasks=[task], verbose=False)        result = crew.kickoff()                self.state.research_output = str(result)                # Assess quality (simulated)        output_length = len(str(result))        if output_length > 500:            self.state.research_quality = 'high'        elif output_length > 200:            self.state.research_quality = 'medium'        else:            self.state.research_quality = 'low'                print(f'  Research complete. Quality: {self.state.research_quality}')        return self.state.research_quality        @listen(conduct_research)    def conditional_analysis(self, quality):        """Step 3: Conditional routing based on research quality."""        print(f'\n[Flow] Step 3: Conditional analysis (quality={quality})')                if quality == 'high':            print('  -> High quality: Proceeding to deep analysis')            return self._deep_analysis()        elif quality == 'medium':            print('  -> Medium quality: Proceeding to standard analysis')            return self._standard_analysis()        else:            print('  -> Low quality: Retrying research')            return self._retry_research()        def _deep_analysis(self):        """Deep analysis for high-quality research."""        analyst = Agent(            role='Senior Analyst',            goal='Provide deep strategic analysis',            backstory='An expert analyst who provides deep insights.',            llm=llm,            verbose=False        )                task = Task(            description=(                f'Provide a deep strategic analysis of: {self.state.research_output}\n\n'                'Include: market implications, competitive analysis, investment outlook.'            ),            expected_output='Deep strategic analysis report.',            agent=analyst        )                crew = Crew(agents=[analyst], tasks=[task], verbose=False)        result = crew.kickoff()        self.state.analysis_output = str(result)        return 'deep'        def _standard_analysis(self):        """Standard analysis for medium-quality research."""        analyst = Agent(            role='Analyst',            goal='Provide standard analysis',            backstory='A competent analyst.',            llm=llm,            verbose=False        )                task = Task(            description=f'Analyze: {self.state.research_output}',            expected_output='Standard analysis report.',            agent=analyst        )                crew = Crew(agents=[analyst], tasks=[task], verbose=False)        result = crew.kickoff()        self.state.analysis_output = str(result)        return 'standard'        def _retry_research(self):        """Retry research with more specific instructions."""        self.state.retry_count += 1        print(f'  Retry attempt #{self.state.retry_count}')                if self.state.retry_count > 2:            self.state.errors.append('Max retries exceeded')            self.state.analysis_output = 'Research failed after multiple attempts.'            return 'failed'                researcher = Agent(            role='Research Specialist',            goal='Provide comprehensive research',            backstory='An expert researcher who provides detailed findings.',            llm=llm,            verbose=False        )                task = Task(            description=(                f'Provide a VERY detailed research report on: {self.state.topic}\n\n'                'Include specific data points, examples, and sources. Minimum 1000 words.'            ),            expected_output='Comprehensive research report.',            agent=researcher        )                crew = Crew(agents=[researcher], tasks=[task], verbose=False)        result = crew.kickoff()        self.state.research_output = str(result)                # Re-assess        output_length = len(str(result))        if output_length > 500:            self.state.research_quality = 'high'        elif output_length > 200:            self.state.research_quality = 'medium'        else:            self.state.research_quality = 'low'                return self.conditional_analysis(self.state.research_quality)        @listen(conditional_analysis)    def generate_final_report(self, analysis_type):        """Step 4: Generate final report."""        print(f'\n[Flow] Step 4: Generating final report (analysis_type={analysis_type})')                writer = Agent(            role='Report Writer',            goal='Create professional reports',            backstory='An expert report writer.',            llm=llm,            verbose=False        )                task = Task(            description=(                f'Write a final report on: {self.state.topic}\n\n'                f'Research: {self.state.research_output}\n\n'                f'Analysis: {self.state.analysis_output}\n\n'                'Format as a professional executive summary.'            ),            expected_output='Professional executive summary report.',            agent=writer        )                crew = Crew(agents=[writer], tasks=[task], verbose=False)        result = crew.kickoff()        self.state.final_report = str(result)                print('  Final report generated')        return self.state.final_reportprint('ResearchFlow class defined with conditional routing and retry logic')

## 6. Execute the Flow

In [ ]:
print('\n' + '=' * 70)print('EXECUTING ADVANCED CREWAI FLOW')print('=' * 70)flow = ResearchFlow()final_report = flow.kickoff()print('\n' + '=' * 70)print('FLOW EXECUTION COMPLETE')print('=' * 70)

## 7. Inspect Flow State

In [ ]:
print('\nFlow State Inspection:')print(f'  Topic: {flow.state.topic}')print(f'  Research Quality: {flow.state.research_quality}')print(f'  Retry Count: {flow.state.retry_count}')print(f'  Errors: {flow.state.errors}')print(f'  Research Output Length: {len(flow.state.research_output)} chars')print(f'  Analysis Output Length: {len(flow.state.analysis_output)} chars')print(f'  Final Report Length: {len(flow.state.final_report)} chars')print('\nFinal Report Preview:')print(flow.state.final_report[:500] + '...')

## 8. Flow Architecture Summary

```
@start() initialize_topic()
       |
       ▼
@listen() conduct_research()
       |
       ▼ (quality assessment)
@listen() conditional_analysis()
       |
       ├── high ──> _deep_analysis()
       ├── medium -> _standard_analysis()
       └── low ───> _retry_research() ──> loop back
       |
       ▼
@listen() generate_final_report()
       |
       ▼
     END
```

---

## ✅ Summary

You have successfully:
1. ✅ Defined a custom Pydantic-based flow state
2. ✅ Created a Flow class with `@start()` and `@listen()` decorators
3. ✅ Implemented conditional routing based on intermediate results
4. ✅ Built retry logic for low-quality outputs
5. ✅ Combined crews as steps within a larger flow
6. ✅ Inspected the final flow state

In the next notebook, we will build the **Production End-to-End CrewAI Project**.